# BlocoRocha A.01: Momentum Growth, annual rebalancing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiago-baptistalopes/bloco-rocha/blob/main/notebooks/A01_momentum_growth.ipynb)

| Item | Rule |
|---|---|
| Universe | 200 mid to large capitalisation firms (100 United States, 100 Europe), sampled at fund creation, 2005-01-25 |
| Entry | Top quartile 12-1 month momentum **and** top half price-to-book; filled to 50 names by combined rank |
| Exit | Debt-to-equity up 50% or more year on year, or price-to-book down 20% or more year on year |
| Rebalancing | Annual (entry, exit checks and equal reweighting) |
| Currency | Euro; United States dollar positions fully hedged (local returns, carry ignored) |
| Persona tests | Maximum drawdown no worse than −35%; value-at-risk-equivalent volatility 12–20% (Summary Risk Indicator class 4) |

Prices: yfinance. Fundamentals: Compustat via Wharton Research Data Services (cached to `data/fundamentals.csv`).

In [ ]:
%pip install -q yfinance quantstats wrds

In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
import quantstats as qs
import matplotlib.pyplot as plt

START = pd.Timestamp('2005-01-25')
HISTORY_START = '2003-12-01'
N_HOLD = 50
N_PER_REGION = 100
MIN_MCAP_EUR = 1.5e9
MOM_Q, PB_Q = 0.75, 0.50
DE_JUMP, PB_DROP = 0.50, -0.20
MDD_LIMIT, VEV_RANGE, RHP_YEARS = -0.35, (0.12, 0.20), 20
SEED = 42
DATA = 'data'
os.makedirs(DATA, exist_ok=True)

## 1. Candidate universe

In [ ]:
US = ['AAPL','MSFT','NVDA','AMZN','ORCL','ADBE','INTU','QCOM','TXN','ADI','AMAT','LRCX','KLAC','MCHP','CSCO','IBM',
      'INTC','MU','AMD','NTAP','CTSH','ACN','ADP','PAYX','FI','EBAY','BKNG','ISRG','ADSK','CDNS','SNPS','AKAM','EA',
      'HPQ','GLW','APH','STX','WDC','AMGN','GILD','BIIB','REGN','VRTX','JNJ','PFE','MRK','ABT','LLY','BMY','MDT',
      'SYK','BSX','BDX','ZBH','TMO','DHR','A','WAT','MTD','IDXX','RMD','EW','UNH','CI','ELV','HUM','CVS','HON','GE',
      'MMM','CAT','DE','EMR','ROK','ETN','PH','ITW','DOV','ROP','AME','GWW','FAST','CMI','PCAR','UNP','CSX','NSC',
      'FDX','UPS','LMT','GD','NOC','RTX','BA','TDY','NKE','SBUX','MCD','HD','LOW','TGT','COST','WMT','TJX','ROST',
      'ORLY','AZO','JPM','BAC','WFC','GS','MS','SCHW','BLK','AXP','XOM','CVX','COP','SLB','HAL','EOG','OXY','PG','KO',
      'PEP','CL','T','VZ','DIS','CMCSA','SHW','ECL','APD','JCI','ADM']
EU = ['ASML.AS','PHIA.AS','HEIA.AS','AD.AS','WKL.AS','INGA.AS','AKZA.AS','KPN.AS','RAND.AS','ASM.AS','BESI.AS',
      'SAP.DE','SIE.DE','ALV.DE','BAS.DE','BAYN.DE','BMW.DE','MBG.DE','VOW3.DE','DTE.DE','EOAN.DE','RWE.DE','MUV2.DE',
      'DBK.DE','ADS.DE','HEN3.DE','IFX.DE','FRE.DE','FME.DE','MRK.DE','BEI.DE','CON.DE','DB1.DE','HEI.DE','PUM.DE',
      'SRT3.DE','CBK.DE','LHA.DE','NEM.DE','RHM.DE','QIA.DE','MC.PA','OR.PA','SAN.PA','TTE.PA','AIR.PA','SU.PA',
      'AI.PA','BNP.PA','GLE.PA','ACA.PA','CS.PA','DG.PA','EN.PA','SGO.PA','CAP.PA','DSY.PA','STMPA.PA','KER.PA',
      'RI.PA','BN.PA','VIE.PA','ORA.PA','ML.PA','HO.PA','EL.PA','RMS.PA','PUB.PA','AC.PA','ATO.PA','SW.PA','TEP.PA',
      'BIM.PA','UBI.PA','ALO.PA','NESN.SW','NOVN.SW','ROG.SW','UBSG.SW','ABBN.SW','ZURN.SW','CFR.SW','SREN.SW',
      'GIVN.SW','SIKA.SW','LONN.SW','GEBN.SW','SGSN.SW','STMN.SW','SOON.SW','LOGN.SW','SCHP.SW','HOLN.SW','TEMN.SW',
      'KNIN.SW','AZN.L','GSK.L','HSBA.L','BP.L','ULVR.L','RIO.L','AAL.L','BATS.L','DGE.L','VOD.L','BARC.L','LLOY.L',
      'PRU.L','RKT.L','REL.L','LSEG.L','AHT.L','BA.L','RR.L','SMIN.L','SN.L','CPG.L','WPP.L','NG.L','TSCO.L','SGE.L',
      'III.L','HLMA.L','SPX.L','CRDA.L','IMB.L','ABF.L','NXT.L','BNZL.L','RTO.L','ITRK.L','LGEN.L','AV.L','STAN.L',
      'NOKIA.HE','ERIC-B.ST','VOLV-B.ST','ATCO-A.ST','SAND.ST','HM-B.ST','SKF-B.ST','ASSA-B.ST','NOVO-B.CO',
      'CARL-B.CO','DSV.CO','COLO-B.CO','MAERSK-B.CO','GN.CO','DEMANT.CO','SAMPO.HE','KNEBV.HE','UPM.HE','EQNR.OL',
      'NHY.OL','TEL.OL','ORK.OL','ITX.MC','SAN.MC','BBVA.MC','IBE.MC','TEF.MC','REP.MC','ENEL.MI','ENI.MI','ISP.MI',
      'UCG.MI','G.MI','TEN.MI','REC.MI','ABI.BR','UCB.BR','SOLB.BR','KBC.BR']
CANDIDATES = US + EU
REGION = pd.Series({t: 'US' for t in US} | {t: 'EU' for t in EU})
len(US), len(EU)

## 2. Prices, currencies and foreign exchange (yfinance)

In [ ]:
def cached(name, fetch, dates=True):
    path = f'{DATA}/{name}.csv'
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0, parse_dates=dates)
    df = fetch()
    df.to_csv(path)
    return df

def fetch_prices():
    return yf.download(CANDIDATES, start=HISTORY_START, auto_adjust=True, progress=False)['Close']

def fetch_meta():
    rows = {}
    for t in CANDIDATES:
        tk = yf.Ticker(t)
        try:
            ccy = tk.fast_info['currency']
        except Exception:
            ccy = None
        try:
            isin = tk.isin if REGION[t] == 'EU' else None
        except Exception:
            isin = None
        rows[t] = {'currency': ccy, 'isin': isin}
    return pd.DataFrame(rows).T

px = cached('prices', fetch_prices).dropna(how='all')
meta = cached('meta', fetch_meta, dates=False)
meta

In [ ]:
def base_ccy(c):
    return 'GBP' if c in ('GBp', 'GBX') else c

CCYS = sorted({base_ccy(c) for c in meta['currency'].dropna()} | {'USD', 'GBP', 'CHF', 'SEK', 'DKK', 'NOK'} - {'EUR'})

def fetch_fx():
    fx = yf.download([f'EUR{c}=X' for c in CCYS], start=HISTORY_START, auto_adjust=True, progress=False)['Close']
    fx.columns = [c[3:6] for c in fx.columns]
    return fx

fx = cached('fx', fetch_fx).reindex(px.index).ffill().bfill()
fx['EUR'] = 1.0

def to_eur(t):
    c = meta.loc[t, 'currency']
    scale = 0.01 if c in ('GBp', 'GBX') else 1.0
    return px[t] * scale / fx[base_ccy(c)]

px_eur = pd.DataFrame({t: to_eur(t) for t in px.columns if pd.notna(meta.loc[t, 'currency'])})
px_ret = pd.DataFrame({t: px[t] if meta.loc[t, 'currency'] == 'USD' else px_eur[t] for t in px_eur.columns})

## 3. Fundamentals (Compustat via Wharton Research Data Services)

United States: `comp.funda`, matched on ticker. Europe: `comp.g_funda`, matched on International Securities Identification Number through `comp.g_security`, with shares outstanding from `comp.g_secd` at fiscal year end.
Debt = long-term debt + debt in current liabilities. Fiscal year $y-1$ is used at the January rebalance of year $y$ (accepted look-ahead bias).

In [ ]:
def fetch_fundamentals():
    import wrds
    db = wrds.Connection()
    us = [t.replace('-', '.') for t in US]
    q_us = f'''
        select tic as ticker, datadate, fyear, curcd, ceq, coalesce(dltt,0)+coalesce(dlc,0) as debt, csho as shares
        from comp.funda
        where indfmt='INDL' and datafmt='STD' and popsrc='D' and consol='C'
          and fyear between 2002 and 2026 and tic in ({",".join(f"'{t}'" for t in us)})'''
    f_us = db.raw_sql(q_us)
    f_us[['ceq', 'debt', 'shares']] *= 1e6
    isins = meta.loc[EU, 'isin'].dropna()
    isins = isins[isins.str.len() == 12]
    q_eu = f'''
        select s.isin, f.datadate, f.fyear, f.curcd, f.ceq, coalesce(f.dltt,0)+coalesce(f.dlc,0) as debt, d.cshoc as shares
        from comp.g_security s
        join comp.g_funda f on f.gvkey = s.gvkey
        left join lateral (
            select cshoc from comp.g_secd x
            where x.gvkey = s.gvkey and x.iid = s.iid and x.datadate <= f.datadate and x.datadate > f.datadate - 10
            order by x.datadate desc limit 1) d on true
        where f.indfmt='INDL' and f.datafmt='HIST_STD' and f.popsrc='I' and f.consol='C'
          and f.fyear between 2002 and 2026 and s.isin in ({",".join(f"'{i}'" for i in isins)})'''
    f_eu = db.raw_sql(q_eu)
    f_eu[['ceq', 'debt']] *= 1e6
    f_eu['ticker'] = f_eu['isin'].map({v: k for k, v in isins.items()})
    db.close()
    f = pd.concat([f_us, f_eu.drop(columns='isin')])
    f['ticker'] = f['ticker'].replace({t.replace('-', '.'): t for t in US})
    return f.sort_values('datadate').drop_duplicates(['ticker', 'fyear'], keep='last').set_index('ticker')

fund = cached('fundamentals', fetch_fundamentals, dates=False).reset_index()
fund = fund[fund['ceq'].notna()]
ceq, debt, shares = (fund.pivot(index='fyear', columns='ticker', values=v) for v in ('ceq', 'debt', 'shares'))
curcd = fund.groupby('ticker')['curcd'].last()
de = (debt / ceq).where(ceq > 0)
fund.groupby('ticker').size().describe()

## 4. Universe at fund creation and annual signals

In [ ]:
pxf = px_eur.ffill(limit=10)
pxr = px_ret.ffill(limit=10)
DATES = [px.index[px.index >= pd.Timestamp(y, 1, 25)][0] for y in range(START.year, px.index[-1].year + 1)
         if (px.index >= pd.Timestamp(y, 1, 25)).any()]

def at(df, d):
    return df.loc[:d].iloc[-1]

def book_eur(d):
    y = d.year - 1
    if y not in ceq.index:
        return pd.Series(dtype=float)
    rate = at(fx, d)
    return ceq.loc[y] / curcd.reindex(ceq.columns).map(rate).astype(float)

def mcap_eur(d):
    y = d.year - 1
    return at(pxf, d).reindex(shares.columns) * shares.loc[y] if y in shares.index else pd.Series(dtype=float)

def pb(d):
    b = book_eur(d)
    return (mcap_eur(d) / b).where(b > 0)

def momentum(d):
    return at(pxr, d - pd.DateOffset(months=1)) / at(pxr, d - pd.DateOffset(months=12)) - 1

mcap0 = mcap_eur(DATES[0]).dropna()
eligible = mcap0[mcap0 >= MIN_MCAP_EUR].index.intersection(at(pxf, DATES[0]).dropna().index)
rng = np.random.default_rng(SEED)
UNIVERSE = []
for r in ('US', 'EU'):
    pool = sorted(t for t in eligible if REGION[t] == r)
    UNIVERSE += list(rng.choice(pool, size=min(N_PER_REGION, len(pool)), replace=False))
pd.Series(REGION[UNIVERSE]).value_counts()

## 5. Portfolio rules

At each annual date $t$: holdings that trigger an exit signal are sold,
$$\frac{D/E_{y-1}}{D/E_{y-2}} - 1 \ge 0.5 \quad \text{or} \quad \frac{P/B_t}{P/B_{t-1}} - 1 \le -0.2,$$
the remaining holdings are kept, and free slots are filled first with names in the top momentum quartile and the top price-to-book half, then with the best remaining combined rank $\tfrac{1}{2}\left(\text{rank}_{\text{momentum}} + \text{rank}_{P/B}\right)$. All holdings are reset to equal weight.

In [ ]:
holdings, log = {}, []
held, prev_pb = [], None
for d in DATES:
    alive = at(pxf, d).reindex(UNIVERSE).dropna().index
    mom, pbs = momentum(d).reindex(alive), pb(d).reindex(alive)
    y = d.year - 1
    de_jump = (de.loc[y] / de.loc[y - 1] - 1 >= DE_JUMP) if {y, y - 1} <= set(de.index) else pd.Series(False, index=de.columns)
    pb_drop = (pbs / prev_pb - 1 <= PB_DROP) if prev_pb is not None else pd.Series(False, index=alive)
    exits = [t for t in held if t not in alive or de_jump.get(t, False) or pb_drop.get(t, False)]
    held = [t for t in held if t not in exits]
    ok = mom.dropna().index.intersection(pbs.dropna().index)
    score = (mom[ok].rank(pct=True) + pbs[ok].rank(pct=True)) / 2
    passed = (mom[ok] >= mom[ok].quantile(MOM_Q)) & (pbs[ok] >= pbs[ok].quantile(PB_Q))
    order = pd.DataFrame({'passed': passed, 'score': score}).sort_values(['passed', 'score'], ascending=False).index
    entries = [t for t in order if t not in held][:max(N_HOLD - len(held), 0)]
    held = held + entries
    holdings[d] = held
    prev_pb = pbs
    log.append({'date': d.date(), 'holdings': len(held), 'entries': len(entries), 'exits': len(exits),
                'passed_both_filters': int(passed.sum()), 'exit_leverage': sum(bool(de_jump.get(t, False)) for t in exits),
                'exit_price_to_book': sum(bool(pb_drop.get(t, False)) for t in exits)})
pd.DataFrame(log).set_index('date')

## 6. Backtest

In [ ]:
def backtest(holdings, prices):
    dates = list(holdings) + [prices.index[-1]]
    parts = []
    for d0, d1 in zip(dates[:-1], dates[1:]):
        p = prices.loc[d0:d1, holdings[d0]].ffill()
        parts.append((p / p.iloc[0]).mean(axis=1).pct_change().dropna())
    return pd.concat(parts)

fund_r = backtest(holdings, px_ret).rename('BlocoRocha A.01')
bench_holdings = {d: list(at(pxf, d).reindex(UNIVERSE).dropna().index) for d in DATES}
bench_r = backtest(bench_holdings, px_ret).rename('Universe equal weight')

cum = (1 + pd.concat([fund_r, bench_r], axis=1)).cumprod()
cum = pd.concat([pd.DataFrame(1.0, index=[DATES[0]], columns=cum.columns), cum])
ax = cum.plot(figsize=(11, 5), logy=True, title='Cumulative returns (start = 1, log scale)')
ax.axhline(1, color='grey', lw=0.8)
plt.show()
cum.iloc[-1]

In [ ]:
qs.reports.metrics(fund_r, benchmark=bench_r, mode='full', display=True)

## 7. Persona tests

Value-at-risk-equivalent volatility (Commission Delegated Regulation (EU) 2017/653, Annex II), with $\sigma$, $\mu_1$ (skewness) and $\mu_2$ (excess kurtosis) of daily log returns, $N = 256\,T$ trading days over the recommended holding period $T$:
$$\text{VaR}_{\text{return}} = \sigma\sqrt{N}\left(-1.96 + 0.474\frac{\mu_1}{\sqrt{N}} - 0.0687\frac{\mu_2}{N} + 0.146\frac{\mu_1^2}{N}\right) - 0.5\,\sigma^2 N, \qquad \text{VEV} = \frac{\sqrt{3.842 - 2\,\text{VaR}_{\text{return}}} - 1.96}{\sqrt{T}}$$

In [ ]:
def vev(r, T=RHP_YEARS):
    x = np.log1p(r.dropna())
    s, m1, m2, N = x.std(), x.skew(), x.kurt(), 256 * T
    var = s * np.sqrt(N) * (-1.96 + 0.474 * m1 / np.sqrt(N) - 0.0687 * m2 / N + 0.146 * m1**2 / N) - 0.5 * s**2 * N
    return (np.sqrt(3.842 - 2 * var) - 1.96) / np.sqrt(T)

def persona_tests(r):
    mdd, v = qs.stats.max_drawdown(r), vev(r)
    return pd.DataFrame({
        'value': [mdd, v, qs.stats.volatility(r), qs.stats.value_at_risk(r)],
        'target': [f'>= {MDD_LIMIT:.0%}', f'{VEV_RANGE[0]:.0%} to {VEV_RANGE[1]:.0%}', 'reference', 'reference (daily, 95%)'],
        'pass': [mdd >= MDD_LIMIT, VEV_RANGE[0] <= v <= VEV_RANGE[1], None, None]},
        index=['Maximum drawdown', 'Value-at-risk-equivalent volatility', 'Annualised volatility', 'Historical value at risk'])

tests = persona_tests(fund_r)
print('Persona conditions met:', bool(tests['pass'].dropna().all()))
tests